# L08 · PE 估值 + 行业对比

**预计时长**：60 min | **难度**：⭐⭐⭐ | **前置**：L07

## 本节目标
1. PE / PB / PS 三大估值指标的含义与适用范围
2. PE 历史分位数（"现在的 PE 比过去 X% 时间低"）
3. 申万一级行业分类
4. `groupby / rank / quantile / cut / qcut` 用法

> ⚠️ akshare 的 PE 数据接口偶尔不稳，本节用一份静态备份的 PE 数据集演示。

## 第 1 段：金融概念

### 三大估值指标
| 指标 | 公式 | 适用 | 缺点 |
|------|------|------|------|
| **PE** (市盈率) | 股价 / 每股收益 | 盈利稳定的公司 | 亏损公司无意义 |
| **PB** (市净率) | 股价 / 每股净资产 | 重资产行业（银行、地产） | 无形资产多的公司失真 |
| **PS** (市销率) | 股价 / 每股营收 | 高增长未盈利公司（电商、SaaS） | 不反映盈利能力 |

### PE 的几种口径
- **静态 PE** (LYR)：用上一年度净利润
- **滚动 PE** (TTM)：用最近 4 个季度净利润（**最常用**）
- **动态 PE** (Forward)：用分析师预测的今年净利润

### PE 分位数（核心概念）
"比亚迪当前 PE 30，处于过去 5 年的 65% 分位" = 过去 5 年有 65% 的时间 PE ≤ 30。
- 分位数 < 20%：**低估**（历史上 80% 时间更贵）
- 20-80%：**合理区间**
- > 80%：**高估**

### 申万行业分类（SW）
A 股最权威的行业分类，分一级（31 个）、二级、三级。例：
- 一级"汽车"含比亚迪、长城汽车、上汽集团
- 一级"传媒"含完美世界、世纪华通
- 一级"食品饮料"含贵州茅台、五粮液

**对比逻辑**：同行业内比 PE 才有意义。茅台 PE 30 不算贵（行业均值 25），但放到银行（行业均值 5）就天价。

In [ ]:
import sys
from pathlib import Path

# 自动定位 phase1_foundation 目录 + project root（兼容两种 jupyter 启动位置）
_cwd = Path.cwd()
_p1 = _cwd if (_cwd / '_data.py').exists() else (_cwd / 'learning' / 'phase1_foundation')
sys.path.insert(0, str(_p1))
_proj = _p1.parent.parent if _p1.name == 'phase1_foundation' else _p1
if (_proj / 'qtrader' / '__init__.py').exists():
    sys.path.insert(0, str(_proj))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import _style
from _data import get_stock_data

In [ ]:
byd = get_stock_data('002594').set_index('date')

## 第 2 段：从价格反推"伪 PE"

真实 PE 需要"每股收益（EPS）"数据。akshare 的 `stock_a_pe_em` 能拉到，但接口不稳定。
本节用**简化教学版**：给三只股票一个假设的 EPS，算 PE。

In [ ]:
# 假设的 EPS（实际项目应从 akshare.stock_financial_abstract 拉）
EPS = {
    "002594": 5.20,   # 比亚迪
    "002602": 0.18,   # 世纪华通
    "002624": 0.45,   # 完美世界
}

# 用三股对齐的 close 价格
from pathlib import Path
panel_path = Path("data/panel_three_stocks.parquet")
if panel_path.exists():
    wide = pd.read_parquet(panel_path)
else:
    frames = {c: get_stock_data(c).set_index('date')['close'] for c in EPS}
    wide = pd.DataFrame(frames).sort_index().ffill()

# 算 PE
pe = wide.copy()
for code, eps in EPS.items():
    pe[code] = wide[code] / eps
pe.tail(3)

## 第 3 段：PE 历史分位数

In [ ]:
def pe_percentile(pe_series: pd.Series, today: pd.Timestamp = None) -> float:
    """计算 PE 在历史区间内的分位数（0~1）。"""
    if today is None:
        today = pe_series.index[-1]
    current = pe_series.loc[:today].iloc[-1]
    history = pe_series.loc[:today].dropna()
    return (history <= current).sum() / len(history)

# 对三股算当前 PE 分位数
for code in pe.columns:
    pct = pe_percentile(pe[code])
    current_pe = pe[code].iloc[-1]
    print(f"{code} 当前 PE {current_pe:>6.2f}, 历史 {pct*100:>5.1f}% 分位")

In [ ]:
# 画比亚迪 PE 历史曲线 + 分位线
fig, ax = plt.subplots(figsize=(12, 4))
pe['002594'].plot(ax=ax, color='steelblue', label='比亚迪 PE')
ax.axhline(pe['002594'].quantile(0.2), color='green', linestyle='--', label='20% 分位（低估）')
ax.axhline(pe['002594'].quantile(0.8), color='red', linestyle='--', label='80% 分位（高估）')
ax.legend()
ax.set_title('比亚迪 PE 历史区间')
plt.show()

## 第 4 段：构造行业 PE 对比数据

模拟一个 10 只股票的"汽车行业" PE 表，演示 `groupby / rank / qcut`。

In [ ]:
# 构造静态示例数据（实战中从 akshare.stock_industry_pe_ratio_cninfo 拉）
auto_industry = pd.DataFrame({
    'code': ['002594', '600104', '600006', '601633', '000625',
             '601238', '600686', '000800', '600178', '000927'],
    'name':  ['比亚迪', '上汽集团', '东风汽车', '长城汽车', '长安汽车',
              '广汽集团', '金龙汽车', '一汽解放', '安凯客车', '中国重汽'],
    'industry': ['汽车'] * 10,
    'pe_ttm':  [28.5, 8.2, 15.3, 22.1, 12.7, 9.8, 18.4, 11.5, 25.6, 10.2],
    'pb':      [4.1, 0.8, 1.5, 2.3, 1.1, 0.9, 1.8, 1.2, 2.8, 1.0],
})
auto_industry

In [ ]:
# rank：在同行业里排名
auto_industry['pe_rank'] = auto_industry['pe_ttm'].rank(ascending=True)  # 小=便宜
auto_industry['pe_pct'] = auto_industry['pe_ttm'].rank(pct=True)
auto_industry = auto_industry.sort_values('pe_pct')
auto_industry[['code', 'name', 'pe_ttm', 'pe_rank', 'pe_pct']]

In [ ]:
# qcut：按分位分箱（如 3 档：便宜/合理/贵）
auto_industry['pe_band'] = pd.qcut(auto_industry['pe_ttm'], q=3,
                                    labels=['低估', '合理', '高估'])
auto_industry[['code', 'name', 'pe_ttm', 'pe_band']]

## 第 5 段：多行业 groupby 聚合

In [ ]:
# 模拟多行业数据
multi = pd.concat([
    auto_industry,
    pd.DataFrame({
        'code': ['600519', '000858', '000568', '002304'],
        'name':  ['贵州茅台', '五粮液', '泸州老窖', '洋河股份'],
        'industry': ['食品饮料'] * 4,
        'pe_ttm':  [30.5, 22.1, 25.3, 18.7],
        'pb':      [11.2, 6.5, 7.8, 4.9],
    }),
    pd.DataFrame({
        'code': ['002602', '002624', '300413', '300251'],
        'name':  ['世纪华通', '完美世界', '芒果超媒', '光线传媒'],
        'industry': ['传媒'] * 4,
        'pe_ttm':  [35.2, 28.5, 22.1, 40.3],
        'pb':      [3.2, 2.5, 3.8, 4.1],
    }),
], ignore_index=True)

# 按行业算 PE 均值/中位数/分位
summary = multi.groupby('industry').agg(
    pe_mean=('pe_ttm', 'mean'),
    pe_median=('pe_ttm', 'median'),
    pb_mean=('pb', 'mean'),
    count=('code', 'count'),
).round(2)
summary

In [ ]:
# 每只股票在自己行业里的 PE 分位
multi['pe_pct_in_industry'] = multi.groupby('industry')['pe_ttm'].rank(pct=True)
multi.sort_values(['industry', 'pe_pct_in_industry'])

## 第 6 段：随堂小练

### 小练：找出"行业里最便宜"的股票
用 `multi` 数据，每个行业挑 PE 分位 < 0.3 的股票（同行业内低估）。

In [ ]:
# TODO: 你的代码（约 2 行）

## 第 7 段：课后练习 + 下节预告

### 📝 `exercises/ex08.py`
1. 写 `pe_percentile(pe_series, lookback_days=252*5)` 带 lookback 参数（默认 5 年）
2. 写 `industry_rank(stocks_df, by='pe_ttm')` 返回带 `rank_in_industry` 列的 DataFrame
3. 写 `value_categories(pe_series, bins=[0, 0.2, 0.8, 1])` 返回每个股票的估值类别（低估/合理/高估）

### 🔮 下节 L09：向量化核心习惯
量化最重要的"心法"。用 `%timeit` 对比 for 循环 vs 向量化，让"100 倍速度差"刻进肌肉记忆。

## 第 8 段：Jupyter tip 🔧
- `pd.qcut(s, q=10)` 分十等分（十分位）—— 量化常用
- `pd.cut(s, bins=[0, 30, 70, 100])` 按自定义区间分箱
- `df.style.background_gradient(cmap='RdYlGn_r')` 给 DataFrame 加渐变背景（看 PE 排名很直观）